# Quick Start
## Install

Clone and install
```bash
git clone https://gitlab.windenergy.dtu.dk/TOPFARM/cuttingedge/pywake/fuga/PyFuga.git
cd PyFuga
pip install -e .
```

or install directly from gitlab

```bash
pip install git+https://gitlab.windenergy.dtu.dk/TOPFARM/cuttingedge/pywake/fuga/PyFuga.git
```

## Make luts

The function `get_luts` takes care of the whole process from input over preLUTs and fourierLUTs to the final LUTs that can be used in PyWake.

PreLUTs and fourierLUTs are loaded from files if present and otherwise generated and saved as netcdf files.

In [ ]:
from pyfuga import get_luts

In [ ]:
luts = get_luts(
    folder="luts",  # Path where all files (intermediate and final) are stored
    zeta0=0,  # Stability parameter
    nkz0=8,  # Number of kz0 per decade. Total number of kz0 = 9 * nkz0 - (nkz0-1)
    nbeta=32,  # Number of beta angles. Total number of beta angles = nbeta + 1
    diameter=80,  # Wind turbine diameter
    zhub=70,  # Wind turbine hub height
    z0=0.00001,  # Roughness length
    zi=400,  # Inversion height
    zlow=70,  # Lower height of output domain. If zlow=zhigh=zhub, the output has only one layer, at hub height
    zhigh=70,  # Upper height of output domain. If zlow=zhigh=zhub, the output has only one layer, at hub height
    lut_vars=["UL"],  # Output data can be any combination of ['UL', 'UT', 'VL', 'VT', 'WL', 'WT', 'PL', 'PT']
    nx=2048,  # Number of points in LUT (U direction)
    ny=512,  # Number of points in LUT (V direction). Note only one half of the domain is stored
    dx=None,
    dy=None,  # Distance between points on the x and y axis
    jit=True,  # If True (default), some slow functions are just-in-time compiled
    n_cpu=None,  # Number of CPUs for parallelization, None means all available CPUs
)

The `luts` folder now contains the following files:
```
.
├── luts 
    ├── preLUTs_Zeta0=0.00_8_32.nc
    ├── fLUTs_Zeta0=0.00_8_32_D80_zhub70_zi400_z0=0.00001000_z70.0_UL.nc
    └── LUTs_Zeta0=0.00_8_32_D80_zhub70_zi400_z0=0.00001000_z70.0_UL_nx2048_ny512_dx20.0_dy5.0.nc
```


In [ ]:
luts

### Hub height normalized deficit field

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 4))
luts.UL.squeeze()[500:600, :100].plot(x="x")
plt.axis("equal")

### Profile 5D downstream

In [ ]:
axes = plt.subplots(1, 2, figsize=(16, 6))[1]
for ax in axes:
    luts.UL.interp(x=80 * 5).plot(ax=ax)
    ax.grid()
axes[1].set_ylim([-0.0025, 0.0025])
axes[0].set_title("Normalized deficit profile 5D downstream")
axes[1].set_title("Zoom of speedup region");

## Input parameters

### nkz0

The `nkz0` parameter determines the wave-number resolution. Lower numbers result in faster computation, smaller prelut files, but also larger wriggles.

Note, for some reason, `nkz0=32` also results in wriggles maybe due to overfitting?

The plot below shows the error compared to `nkz0=16` in the cross-wind profile 5D downstream. In the left plot, the error is shown relative to the maximum wake value, and `nkz0=8` seems to be sufficient.

In the right plot, the error is shown relative to the maximum speed-up, and here `nkz0=8` deviates more than 0.25% from `nkz0=16`. Hence, `nkz0=16` may be needed for studies where accurate speedup effects are important.

![nkz0.png](../_static/nkz0.png)

